In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
import glob
DATASET_PATH = 'D:\\uni - 4th sem\\uni 4th\\CV\\FinalProjectCV_Kelompok03\\images'

malam_files = glob.glob(os.path.join(DATASET_PATH, "*Cv Malam*.*"))
pagi_files = glob.glob(os.path.join(DATASET_PATH, "*Cv Pagi*.*"))
siang_files = glob.glob(os.path.join(DATASET_PATH, "*Cv Siang*.*"))

print(f"Total gambar Cv Malam: {len(malam_files)}")
print(f"Total gambar Cv Pagi: {len(pagi_files)}")
print(f"Total gambar Cv Siang: {len(siang_files)}")

# ROI using Mouse Click

In [ ]:
import cv2
import numpy as np

# Global list to store clicked points
points = []

def click_event(event, x, y, flags, param):
    global points
    
    # Capture left mouse clicks
    if event == cv2.EVENT_LBUTTONDOWN:
        points.append((x, y))
        
        # Draw a small circle where the user clicked
        cv2.circle(image_copy, (x, y), 5, (0, 0, 255), -1)
        
        # Draw lines connecting the points
        if len(points) >= 2:
            cv2.line(image_copy, points[-2], points[-1], (0, 255, 0), 2)
            
        cv2.imshow("Define Polygon ROI", image_copy)

# Load image and create a copy for drawing
image = cv2.imread(malam_files[0])
image_copy = image.copy()

cv2.imshow("Define Polygon ROI", image_copy)
# Attach the mouse callback function to the window
cv2.setMouseCallback("Define Polygon ROI", click_event)

print("Click points to define the ROI. Press 'q' when finished.")

# Wait until 'q' is pressed
while True:
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

# If points were selected, close the polygon and apply the mask
if len(points) > 2:
    # Connect the last point to the first point
    cv2.line(image_copy, points[-1], points[0], (0, 255, 0), 2)
    cv2.imshow("Define Polygon ROI", image_copy)
    cv2.waitKey(500) # Pause briefly to show the closed shape
    
    # Create a black mask of the same size as the image
    mask = np.zeros(image.shape[:2], dtype=np.uint8)
    
    # Fill the polygon on the mask with white (255)
    polygon = np.array([points], dtype=np.int32)
    cv2.fillPoly(mask, polygon, 255)
    
    # Apply the mask to the original image using bitwise_and
    masked_image = cv2.bitwise_and(image, image, mask=mask)
    
    cv2.imshow("Final ROI", masked_image)
    cv2.waitKey(0)

cv2.destroyAllWindows()

In [ ]:
img_path= r'D:\uni - 4th sem\uni 4th\CV\FinalProjectCV_Kelompok03\images\Cv Malam_frame_0000.png'
img = cv2.imread(img_path)
img.shape 

# Modelling

In [14]:
# langsung masuk YOLO

import cv2
import random
from ultralytics import YOLO

def getColours(cls_num):
    """Generate unique colors for each class ID"""
    random.seed(cls_num)
    return tuple(random.randint(0, 255) for _ in range(3))

model = YOLO('yolo11n.pt')  # 

video_path = "D:\\uni - 4th sem\\uni 4th\\CV\\FinalProjectCV_Kelompok03\\dataset\\Cv Siang.mp4"
videoCap = cv2.VideoCapture(video_path)

frame_count = 0

while True:
    ret, frame = videoCap.read()
    if not ret:
        break
    results = model.track(frame, stream=True)

    for result in results:
        class_names = result.names
        for box in result.boxes:
            if box.conf[0] > 0.4:
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                cls = int(box.cls[0])
                class_name = class_names[cls]

                conf = float(box.conf[0])

                colour = getColours(cls)

                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)

                cv2.putText(frame, f"{class_name} {conf:.2f}",
                            (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, colour, 2)

    if frame_count < 20:
        cv2.imshow('Frame', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            # increment frame counter and allow quitting with 'q'
            frame_count += 1
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
videoCap.release()
cv2.destroyAllWindows()


0: 384x640 7 persons, 2 cars, 6 motorcycles, 2 traffic lights, 106.8ms
Speed: 2.7ms preprocess, 106.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 3 cars, 6 motorcycles, 2 traffic lights, 104.4ms
Speed: 2.2ms preprocess, 104.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 3 cars, 6 motorcycles, 1 traffic light, 104.7ms
Speed: 3.0ms preprocess, 104.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 3 cars, 6 motorcycles, 1 traffic light, 93.7ms
Speed: 2.5ms preprocess, 93.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 3 cars, 6 motorcycles, 1 traffic light, 102.6ms
Speed: 2.3ms preprocess, 102.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 3 cars, 7 motorcycles, 1 traffic light, 102.2ms
Speed: 2.4ms preprocess, 102.2ms inference, 1.8ms postprocess per image at shape (1,